<a href="https://colab.research.google.com/github/antoniolopez02-oss/AAI2025/blob/2026Fall/customer_churn_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/antoniolopez02-oss/AAI2025/blob/dev/ML/customer_churn_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

            # Part 2: Customer Churn Prediction


This notebook uses logistic regression to estimate the probability that a
telecommunications customer will leave. It uses the 7,043-record
[IBM Telco Customer Churn dataset](https://github.com/IBM/watsonx-ai-samples/blob/master/cloud/data/customer_churn/WA_FnUseC_TelcoCustomerChurn.csv).

Run the cells from top to bottom in Google Colab.

## 1. Load and clean the customer data

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Data source: IBM watsonx-ai-samples, Telco Customer Churn dataset.
# https://github.com/IBM/watsonx-ai-samples/blob/master/cloud/data/customer_churn/WA_FnUseC_TelcoCustomerChurn.csv
DATA_FILENAME = "telco_customer_churn.csv"
DATA_URL = (
    "https://raw.githubusercontent.com/antoniolopez02-oss/"
    "AAI2025/dev/ML/telco_customer_churn.csv"
)
NUMERIC_FEATURES = ["tenure", "MonthlyCharges", "TotalCharges"]
CATEGORICAL_FEATURES = ["Contract", "InternetService", "PaymentMethod"]
TARGET = "Churn"
REQUIRED_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES + [TARGET]

local_paths = [Path(DATA_FILENAME), Path("ML") / DATA_FILENAME]
data_source = next((path for path in local_paths if path.exists()), DATA_URL)
data = pd.read_csv(data_source)

missing_columns = set(REQUIRED_COLUMNS) - set(data.columns)
if missing_columns:
    raise ValueError(f"Missing columns: {sorted(missing_columns)}")

data = data[REQUIRED_COLUMNS].copy()
for column in NUMERIC_FEATURES:
    data[column] = pd.to_numeric(data[column], errors="coerce")
data[TARGET] = data[TARGET].map({"Yes": 1, "No": 0})
data = data.dropna()

if len(data) < 100 or data[TARGET].nunique() != 2:
    raise ValueError("The cleaned dataset does not meet the assignment requirements.")

print(f"Valid customer records loaded: {len(data):,}")
print(f"Observed churn rate: {data[TARGET].mean():.1%}")
data.head()

## 2. Scale, encode, and train the model

`StandardScaler` places the numerical features on comparable scales.
`OneHotEncoder` converts the categorical features into numerical columns.

In [ ]:
features = data[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
target = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.20,
    random_state=42,
    stratify=target,
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), NUMERIC_FEATURES),
        (
            "categorical",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=False,
            ),
            CATEGORICAL_FEATURES,
        ),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)
model.fit(X_train, y_train)
print("Model training complete.")

## 3. Evaluate the model and predict churn risk

In [ ]:
threshold = 0.50
test_probabilities = model.predict_proba(X_test)[:, 1]
test_predictions = (test_probabilities >= threshold).astype(int)

print(
    f"Test accuracy at the {threshold:.2f} threshold: "
    f"{accuracy_score(y_test, test_predictions):.1%}"
)
print(f"Test ROC AUC: {roc_auc_score(y_test, test_probabilities):.3f}")

new_customer = pd.DataFrame(
    {
        "tenure": [12],
        "MonthlyCharges": [85.00],
        "TotalCharges": [1020.00],
        "Contract": ["Month-to-month"],
        "InternetService": ["Fiber optic"],
        "PaymentMethod": ["Electronic check"],
    }
)

churn_probability = model.predict_proba(new_customer)[0, 1]
churn_prediction = int(churn_probability >= threshold)

print(f"\nNew customer churn probability: {churn_probability:.1%}")
print(f"Churn prediction (1 = at risk, 0 = not at risk): {churn_prediction}")
print(
    f"Interpretation: The model estimates a {churn_probability:.1%} "
    "chance that this customer will leave."
)

if churn_prediction == 1:
    print(
        "Business use: Flag this customer for a retention offer, "
        "service review, or a more suitable contract."
    )
else:
    print("Business use: Continue normal support and monitor future risk.")

## 4. Print and explain the coefficients

In [ ]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["classifier"].coef_[0]
coefficient_pairs = [
    (
        name.replace("numeric__", "").replace("categorical__", ""),
        coefficient,
    )
    for name, coefficient in zip(feature_names, coefficients)
]
coefficient_pairs.sort(key=lambda item: abs(item[1]), reverse=True)

print("Model coefficients:")
for feature, coefficient in coefficient_pairs:
    print(f"  {feature}: {coefficient:,.3f}")

print("\nCoefficient explanation:")
print("  Positive coefficients increase predicted churn likelihood.")
print("  Negative coefficients decrease predicted churn likelihood.")
print(
    "  Numerical coefficients represent a one-standard-deviation change "
    "because those features were scaled."
)
print(
    "  Categorical coefficients compare each displayed category with "
    "the category dropped by OneHotEncoder."
)